In [2]:
# =============================================================================
# Wavevector (k) Filtering of mumax3 Spin-Wave Simulation Results
# =============================================================================
# This notebook post-processes frequency-domain magnetization data (.npz files
# produced by preprocessing.ipynb) to isolate individual spin-wave modes by
# their wavevector. The workflow for each simulation file is:
#   1. Load the pre-computed FFT data at a target frequency (confluence or splitting).
#   2. Extract a 1-D line profile at the sample edge and compute its spatial FFT
#      to identify the dominant ky wavevector of the scattered spin wave.
#   3. Compute the full 2-D spatial FFT (kx-ky map) of the magnetisation pattern.
#   4. Apply a Gaussian mask in k-space centred on the target (kx, ky) peak to
#      suppress all other wavevector contributions (i.e. the k-filter).
#   5. Inverse-FFT the masked spectrum back to real space, yielding the isolated
#      spin-wave amplitude map.
#   6. Save the filtered wave and its spatial coordinates to a compressed .npz file.
# Two physical processes are handled separately:
#   - Confluence : f_beam + f_edge  (sum-frequency generation)
#   - Splitting  : f_beam - f_edge  (difference-frequency generation)
# =============================================================================

# --- Standard scientific libraries ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors          # Extended colormap utilities
import matplotlib.patches as patches        # Shape primitives for annotations
import scipy.optimize as opt
from scipy.signal import find_peaks         # Peak detection in 1-D spectra
from matplotlib.colors import LogNorm       # Logarithmic colour normalisation for imshow

import glob

# --- mumax3 post-processing library ---
import mumax3PP.ovf as ovf                             # Reader for mumax3 .ovf / .npz output
import mumax3PP.parameters as parameters               # Simulation parameter configuration
import mumax3PP.fft_across_xyzm as FFT_across_xyzm    # Multi-threaded FFT utilities

# --- Convenience aliases for common math functions ---
sqrt = np.sqrt
pi   = np.pi
exp  = np.exp
sin  = np.sin
cos  = np.cos
mu0  = pi * 4e-7    # Vacuum permeability (H/m)


In [3]:
# =============================================================================
# Cell 2: Helper functions & material parameters for the dispersion relation
# =============================================================================

def findNearest(array, value):
    """
    Return the index and value of the element in `array` closest to `value`.

    Parameters
    ----------
    array : array-like
        The array to search.
    value : float
        The target value.

    Returns
    -------
    idx : int
        Index of the nearest element.
    array[idx] : float
        The nearest element value.
    """
    idx = (np.abs(array - value)).argmin()
    return idx, array[idx]


def kalDR(k, angle_z, angle_k, d, mu0H0eff, wM, lb, mu0Ha, gamma):
    """
    Kalinikos-Slavin spin-wave dispersion relation (simplified form).

    Computes the spin-wave frequency for a thin magnetic film using the
    Kalinikos-Slavin formalism. This version uses a simplified expression
    for the dipolar matrix element F.

    Parameters
    ----------
    k         : float or array  Wavevector magnitude (rad/m).
    angle_z   : float           Polar angle of magnetisation w.r.t. film normal (rad).
    angle_k   : float           Azimuthal angle of k w.r.t. applied field (rad).
    d         : float           Film thickness (m).
    mu0H0eff  : float           Effective field mu0*Heff (T).
    wM        : float           Characteristic frequency gamma*mu0*Ms (rad/s).
    lb        : float           Exchange length parameter (m^2*rad/s).
    mu0Ha     : float           Anisotropy field mu0*Ha (T).
    gamma     : float           Gyromagnetic ratio (rad/(s·T)).

    Returns
    -------
    float or array
        Spin-wave frequency in Hz.
    """
    def Fcoef1(k, theta, phi, w0):
        """Dipolar matrix element F (simplified, includes surface pinning factor P)."""
        P = 1 - (1 - exp(-k * d)) / (k * d)   # Dipolar pinning factor for finite-thickness film
        return P + (sin(theta)**2) * (1 - P * (1 + cos(phi)**2) +
                   sin(phi)**2 * (wM / w0) * (1 - exp(-2 * k * d)) / 4)

    w0 = gamma * mu0H0eff + gamma * mu0Ha + gamma * lb * k**2   # FMR frequency + exchange shift
    w2 = w0 * (w0 + wM * Fcoef1(k, angle_z, angle_k, w0))      # Squared angular frequency
    return np.sqrt(w2) / (2 * np.pi)                             # Convert to Hz


def gedDT(times):
    """
    Estimate the simulation time step dt from an array of output timestamps
    via a robust linear least-squares fit.

    See `getDT` in preprocessing.ipynb for a detailed explanation.

    Parameters
    ----------
    times : array-like  Simulation timestamps (s).

    Returns
    -------
    float  Fitted time step dt (s).
    """
    xdata = range(len(times))
    from scipy.optimize import curve_fit
    def func(x, a, b):
        return a * x + b
    dt = np.abs(times[-1] - times[1]) / (len(times) - 1)   # Initial estimate
    popt, pcov = curve_fit(func, xdata, times, p0=[dt, 0])
    # Diagnostic plots (commented out):
    #     plt.plot(xdata, times)
    #     plt.plot(xdata, func(xdata, *popt))
    return popt[0]


def kalDR_full(k, angle_k, angle_z, d, mu0H0eff, wM, lb, mu0Ha, gamma):
    """
    Kalinikos-Slavin dispersion relation (full form).

    More complete version of `kalDR` using the full dipolar matrix element
    that correctly accounts for the P*(1-P) cross term.

    Parameters are the same as `kalDR` (note: angle_k and angle_z are swapped
    in the argument list relative to `kalDR`).

    Returns
    -------
    float or array  Spin-wave frequency in Hz.
    """
    mu0 = np.pi * 4e-07
    w0 = gamma * mu0H0eff + gamma * mu0Ha + gamma * lb * k**2

    def Fcoef1(k, theta, phi, w0):
        """Full dipolar matrix element including the P*(1-P) exchange-dipolar cross term."""
        P = 1 - (1 - exp(-k * d)) / (k * d)
        return P + (sin(theta)**2) * (1 - P * (1 + cos(phi)**2) +
                   sin(phi)**2 * P * (1 - P) * wM / w0)

    w2 = w0 * (w0 + wM * Fcoef1(k, angle_z, angle_k, w0))
    return np.sqrt(w2) / (2 * np.pi)


# --- Material parameters for Permalloy (Py, Ni80Fe20) ---
Ms    = 800e3      # Saturation magnetisation (A/m)
Aex   = 13e-12     # Exchange stiffness constant (J/m)
gamma = 176e09     # Gyromagnetic ratio (rad/(s·T))
Lz    = 10e-09     # Film thickness (m)
mu0Heff = 0.3      # Effective applied field mu0*Heff (T) — includes demagnetisation
mu0Ha   = 0.0      # Uniaxial anisotropy field (T); zero for Py

# --- Derived magnetic parameters ---
wM   = gamma * mu0 * Ms             # Angular frequency scale wM = gamma * mu0 * Ms (rad/s)
w0   = gamma * mu0Heff + gamma * mu0Ha   # FMR frequency (rad/s), without exchange
lex2 = 2 * Aex / (mu0 * Ms**2)     # Squared exchange length lex^2 (m^2)
lb   = mu0 * Ms * lex2              # Exchange parameter lb used in dispersion (m^2·rad/s)

# --- Geometry: magnetisation aligned in-plane (phi_z = 90°) ---
phi_z = 90 * (np.pi / 180)          # Polar angle of equilibrium magnetisation (rad)
# phi_H = pi/2                      # (Commented) Applied field in-plane
parametersPy = [phi_z, Lz, mu0Heff, wM, lb, mu0Ha, gamma]   # Packed parameter list for DR calls

# --- Precompute the 2-D dispersion relation over a k-space grid ---
# This reference surface fqs_2d(kx, ky) is used as an overlay on k-space plots
# to verify that identified peaks lie on the theoretical dispersion contours.
k_max = 4e8    # Maximum wavevector for the grid (rad/m)

# Build a symmetric 1-D wavevector array (negative and positive halves)
k_tmp = np.linspace(1, k_max, 250)
k_tmp = np.concatenate((-k_tmp[::-1], k_tmp))   # Symmetric: [-k_max ... -1, 1 ... k_max]

# 2-D meshgrid of (kx, ky) for the dispersion surface
kxx, kyy = np.meshgrid(k_tmp, k_tmp)

angle_z = pi / 2     # Magnetisation in-plane (polar angle = 90°)

# Pre-allocate arrays for the dispersion surface and intermediate quantities
f      = np.zeros_like(kxx)
km     = np.zeros_like(kxx)       # |k| = sqrt(kx^2 + ky^2) at each grid point
phi_Hm = np.zeros_like(kxx)       # Azimuthal angle of k w.r.t. x-axis at each grid point

# Compute |k| and phi(k) at every grid point
for i in range(kyy.shape[0]):
    for j in range(kyy.shape[1]):
        phi_Hm[i, j] = np.arctan(kyy[i, j] / kxx[i, j])           # atan(ky/kx)
        km[i, j]     = np.sqrt(kxx[i, j]**2 + kyy[i, j]**2)       # |k|

# Evaluate the full Kalinikos-Slavin dispersion at every (kx, ky) point
parameters2 = [Lz, mu0Heff, wM, lb, mu0Ha, gamma]   # Alternative parameter pack (kept for reference)
fqs_2d = kalDR_full(km, phi_Hm, *parametersPy)       # Shape: (N_k, N_k), values in Hz


In [4]:
# --- Global plotting / simulation constants ---
f_SWB = 45                              # Spin-wave beam (SWB) frequency in GHz (45 GHz carrier)
plt.rcParams.update({'font.size': 8})   # Set default font size for all figures


In [5]:
# --- Default single-run parameters (overridden in batch loops below) ---
amp   = "0.0300"   # Excitation amplitude (string, matches filename convention)
angle = 40         # Beam incidence angle (degrees)
f     = 11.0       # Edge excitation frequency (GHz)


In [6]:
# =============================================================================
# Cell 6: Discover available pre-processed FFT files and extract frequency list
# =============================================================================
# Scan the output directory for .npz files matching the naming pattern for a
# given beam angle and excitation amplitude. The edge excitation frequency is
# parsed directly from each filename and collected into the array `ff`.
# This is useful for verifying which frequencies have been pre-processed
# before running the batch filtering loop.

amp   = "0.0300"
angle = "30"       # Beam angle (string for path formatting)
ff    = []         # List to collect edge frequencies found on disk

# Glob for all Mz FFT files at this angle/amplitude combination
dirLi = glob.glob(r"D:\mumax3\inelastic_scattering\k_neg\{}\{}\B_300mT_*{}_Mz_fzyxm.npz".format(angle, amp, amp))
dirLi = sorted(dirLi)   # Sort to ascending frequency order

for i in dirLi:
    # Extract the edge frequency from the filename.
    # Filename structure: ...edge_f_<FREQ>GHz_amp_...
    # split('\\')[-1]  → last path component (filename)
    # split('_')[-5]   → field 5 from the end = '<FREQ>GHz'
    # [:-3]            → strip 'GHz' suffix → float-parseable string
    ff.append(float(i.split("\\")[-1].split("_")[-5][:-3]))
    print(i)

ff = np.array(ff)   # Convert to NumPy array for downstream indexing


In [7]:
# Display the extracted frequency array for a quick sanity check
ff


In [8]:
# Define the list of edge excitation frequencies (GHz) to process in the batch loop.
# This uniform grid covers 11.0–15.5 GHz in 0.5 GHz steps.
# For angle-specific subsets, see the commented blocks in the cell below.
f_Li = np.arange(11.0, 15.51, 0.5, dtype=float)


In [ ]:
# =============================================================================
# Cell: Confluence (f_beam + f_edge) — k-space filtering batch loop
# =============================================================================
# For each combination of beam angle and edge excitation frequency, this cell:
#   1. Loads the pre-computed FFT data and extracts spatial coordinates.
#   2. Selects the magnetisation snapshot at the confluence frequency
#      f_conf = f_SWB + f_edge (sum-frequency process).
#   3. Identifies the dominant ky of the scattered wave from a 1-D edge profile FFT.
#   4. Computes the 2-D spatial FFT (k-space map) and locates the kx peak.
#   5. Applies a 2-D Gaussian mask to isolate the target (kx, ky) component.
#   6. Inverse-FFTs back to real space and saves the filtered wave.
# =============================================================================

# --- Batch control parameters ---
# Active set overrides the broader lists defined above; edit here to change scope.
angle_Li = np.array([30, 35, 40, 45])              # Full angle sweep (degrees)
f_Li     = np.arange(11.0, 15.51, 0.5, dtype=float)  # Full frequency sweep (GHz)
# f_Li = [  15.0  ]                                  # Single-frequency debug run

# Active (narrowed) parameter set for this run
angle_Li = np.array([32.5])                         # Single angle for this batch
f_Li     = ["11.00", "11.50", "12.00", "12.50"]     # String list (matches filename format exactly)
k_dire   = 'k_pos'   # Wavevector direction folder: 'k_pos' or 'k_neg'

# Accumulators for the identified wavevector components across all files
Kx_vec_array = []   # kx of the scattered spin wave (rad/µm)
Ky_vec_array = []   # ky of the scattered spin wave (rad/µm)
wavelength   = []   # Total wavelength lambda = 2pi/|k| (m)


for angle in angle_Li:

    for f in f_Li:

        # --- Build file paths ---
        dir0     = r"D:\mumax3\inelastic_scattering\{}\{}\{}".format(k_dire, angle, amp)
        data     = r"\B_300mT_beam_f_45GHz_angle_{}_deg_edge_f_{}GHz_amp_{}_Mz_fzyxm.npz".format(angle, f, amp)
        dir_save = r'D:\mumax3\inelastic_scattering\{}\figs\{}\\'.format(k_dire, angle)
        path     = dir0 + data

        print("Opening: " + path)
        print("...")

        # --- Load FFT data and extract spatial grid ---
        M_fzyxm = ovf.OvfFile(path)
        M_fzyxm.array /= (M_fzyxm.array).shape[0]   # Normalise by number of frequency bins
        M   = M_fzyxm.array     # Shape: (N_freq, nz, ny, nx, n_comp)
        fqs = M_fzyxm.time      # Frequency axis (Hz) from the FFT step

        # Physical cell sizes from the simulation header (metres)
        cy = (M_fzyxm._headers)['ystepsize']
        cx = (M_fzyxm._headers)['xstepsize']

        # Real-space coordinate arrays (metres)
        y = np.arange(0, np.shape(M_fzyxm.array)[2], 1) * cy
        x = np.arange(0, np.shape(M_fzyxm.array)[3], 1) * cx


        # ── Step 1: select the confluence frequency slice ────────────────────
        # Confluence = sum-frequency process: f_conf = f_SWB + f_edge
        # We find the closest frequency bin in the pre-computed FFT data.
        freq      = f_SWB * 1e9 + float(f) * 1e9    # Target frequency (Hz)
        print(freq / 1e9)                             # Print in GHz for verification

        f_SWB_id = findNearest(fqs, freq)[0]          # Index of nearest frequency bin
        M_SWB    = M[f_SWB_id, 0, :, :, 0]           # 2-D Mz slice at confluence frequency; shape (ny, nx)
        colormap = np.abs(M_SWB)                      # Amplitude for plotting

        # Spatial crop window (full extent here; x0:x1 and y0:y1 can be narrowed)
        x0, x1 = 0, -1
        y0, y1 = 0, -1

        # --- Plot unfiltered real-space amplitude at confluence frequency ---
        plt.figure(figsize=(6, 6))
        plt.title("Confluence: " + data[:-23])
        im = plt.imshow(colormap[y0:y1, x0:x1], aspect='auto', origin="lower",
                        extent=[x[x0:x1].min() * 1e6, x[x0:x1].max() * 1e6,
                                y[y0:y1].min() * 1e6, y[y0:y1].max() * 1e6],
                        cmap="magma")
        plt.colorbar(im, shrink=0.3)
        plt.xlabel(r"x ($\mathrm{\mu m}$)")
        plt.ylabel(r"y ($\mathrm{\mu m}$)")
        # plt.savefig(...)   # Uncomment to save figure to disk
        plt.show()


        # ── Step 2: 1-D edge profile → dominant ky identification ────────────
        # Extract the real part of Mz along the right edge (x = -1) as a function of y.
        # A 1-D FFT of this line scan yields the transverse wavevector ky of the
        # scattered spin wave propagating along the edge.
        # Note: the if/else branches are currently identical; they allow angle-specific
        # treatment (e.g. selecting a different edge column) to be added later.
        if angle == 30:
            signal = np.real(M_SWB[:, -1])
        else:
            signal = np.real(M_SWB[:, -1])

        # Plot the 1-D edge profile in real space (y vs. Mz)
        plt.title(path.split("\\")[-1][:-13])
        plt.plot(y * 1e6, signal, c="b", lw=0.5)
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        # 1-D FFT of the edge profile → wavevector spectrum along y
        fft   = np.fft.fft(signal)
        k_vec = 2 * np.pi * np.fft.fftfreq(signal.shape[-1], y[1] - y[0])   # ky axis (rad/m)
        sh    = fft.shape[0] // 2   # Only positive frequencies are meaningful (real input)

        # Detect peaks in the positive-ky half of the spectrum.
        # Height threshold at 99% of maximum isolates only the dominant peak.
        if f == 15.0:
            peaks, _ = find_peaks(np.abs(fft)[:sh], height=np.abs(fft)[:sh].max() * 0.99)
        else:
            peaks, _ = find_peaks(np.abs(fft)[:sh], height=np.abs(fft)[:sh].max() * 0.99)

        # Plot the 1-D k-spectrum with peak positions annotated
        plt.title(path.split("/")[-1][:-23])
        plt.plot(k_vec[:sh] * 1e-6, np.abs(fft)[:sh], c="b", lw=0.5)
        plt.xlim(0, 300)
        nn = 1
        for i in range(np.shape(peaks)[0]):
            plt.axvline(k_vec[peaks[i]] * 1e-6, c="k", ls="--", lw=0.5,
                        label="Peak {}. = ".format(nn) + r"{}".format(
                            round(k_vec[peaks[i]] * 1e-6, 2)) + r" (rad/$\mathrm{\mu}$m)")
            nn += 1
        plt.legend()
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        print(k_vec[peaks])   # Print the identified peak ky values (rad/m)


        # --- Also visualise the 2-D amplitude map for reference ---
        im = plt.imshow(np.abs(M_SWB[y0:y1, x0:x1]), aspect='auto', origin="lower",
                        extent=[x[x0:x1].min() * 1e6, x[x0:x1].max() * 1e6,
                                y[y0:y1].min() * 1e6, y[y0:y1].max() * 1e6],
                        cmap="magma")
        plt.colorbar(im, shrink=0.3)
        plt.xlabel(r"x ($\mathrm{\mu m}$)")
        plt.ylabel(r"y ($\mathrm{\mu m}$)")
        plt.show()


        # ── Step 3: 2-D spatial FFT → k-space map ───────────────────────────
        # Compute the 2-D FFT of the cropped Mz slice. The result is fft-shifted
        # so that (kx=0, ky=0) appears at the centre of the 2-D array.
        # Note: if/else branches are identical; angle-specific treatment can be added.
        if angle == 30:
            fft_i = np.fft.fft2(M_SWB[y0:y1, x0:x1])
        else:
            fft_i = np.fft.fft2(M_SWB[y0:y1, x0:x1])

        fft = np.fft.fftshift(fft_i, axes=(0, 1))   # Centre the zero-frequency component
        DR  = np.abs(fft)                            # 2-D spectral amplitude (dispersion relation map)

        # Build the physical kx and ky axes corresponding to the shifted FFT
        x_Li = x[x0:x1];  y_Li = y[y0:y1]
        dx   = x_Li[1] - x_Li[0];  dy = y_Li[1] - y_Li[0]

        # fftfreq returns frequencies in the standard (unshifted) order;
        # manual roll by N/2 reproduces the fftshift ordering for the axis labels.
        kxx_i = np.fft.fftfreq(len(x_Li), dx) * 2 * pi
        kxx_i = np.concatenate((kxx_i[kxx_i.shape[0] // 2:], kxx_i[:kxx_i.shape[0] // 2]))
        kyy_i = np.fft.fftfreq(len(y_Li), dy) * 2 * pi
        kyy_i = np.concatenate((kyy_i[kyy_i.shape[0] // 2:], kyy_i[:kyy_i.shape[0] // 2]))


        # ── Step 4: locate (kx, ky) of the dominant scattered wave ───────────
        mini = 1e-3          # LogNorm lower bound for plotting
        maxi = 0.5 * DR.max()  # LogNorm upper bound (50% of peak)
        d    = 1e6           # Conversion factor: rad/m → rad/µm for axis labels
        acc  = 10e6          # Gaussian filter half-width in k-space (rad/m); ~10 rad/µm

        k0 = 0   # Index selecting the first (dominant) detected peak from the 1-D FFT

        # Define the ky filter window around the detected edge-profile peak
        kycut0 = -1 * k_vec[peaks[k0]] - acc   # Lower ky boundary (rad/m)
        kycut1 = -1 * k_vec[peaks[k0]] + acc   # Upper ky boundary (rad/m)
        # Centre ky value in µm units (for plot title)
        KY = round((np.abs(kycut0) + np.abs(kycut1)) / 2e6, 2)

        # Find the kx position of the spectral maximum along the ky = k_peak row
        cut    = DR[findNearest(kyy_i, -1 * k_vec[peaks[k0]])[0], :]  # 1-D cut at ky = k_peak
        maxi   = findNearest(cut, cut.max())[0]   # Index of kx maximum in that row
        k_maxi = kxx_i[maxi]                      # kx value at the spectral peak (rad/m)

        # Record the identified wavevector components (in rad/µm)
        Ky_vec_array.append(k_vec[peaks[0]] * 1e-6)
        Kx_vec_array.append(k_maxi * 1e-6)

        kxcut0 = k_maxi - acc   # Lower kx boundary for the filter window (rad/m)
        kxcut1 = k_maxi + acc   # Upper kx boundary for the filter window (rad/m)

        # Dispersion contour levels to overlay: carrier frequency and confluence frequency
        freqLevels = [45, freq / 1e9]

        # --- Plot the 2-D k-space amplitude map with dispersion contours ---
        plt.figure(figsize=(6, 6))
        plt.title(r"confluence, f+$\nu$ = " + f"{round(freq / 1e9, 3)} GHz")
        im = plt.imshow(DR, origin="lower", aspect="auto",
                        norm=LogNorm(vmin=1e-3, vmax=0.5 * DR.max()),
                        extent=[np.amin(kxx_i) / d, np.amax(kxx_i) / d,
                                np.amin(kyy_i) / d, np.amax(kyy_i) / d])
        # Dispersion contours (commented out for confluence; the carrier and scattered
        # wave lie on different branches so overlaying both can be visually cluttered)
        # CS = plt.contour(kxx/d, kyy/d, fqs_2d*1e-09, colors='red',
        #                  levels=freqLevels, linestyles='--', linewidths=0.5)
        plt.xlim(-250, 250);  plt.ylim(-250, 250)
        plt.xlabel(r"$k_{x}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        plt.ylabel(r"$k_{y}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        # Crosshair lines marking the identified (kx, ky) peak
        plt.axhline(-1 * k_vec[peaks[k0]] / d, c="magenta", lw=1.5, ls="--")
        plt.axvline(k_maxi / d,                c="magenta", lw=1.5, ls="--")
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        # Accumulate the total wavelength of the scattered spin wave: lambda = 2pi/|k|
        wavelength.append((2 * np.pi) / np.sqrt(k_vec[peaks[k0]]**2 + k_maxi**2))


        # ── Step 5: apply 2-D Gaussian k-space mask ──────────────────────────
        # The mask suppresses all k-space components except those in a narrow
        # Gaussian window centred on the identified (kx_peak, ky_peak).
        # The mask is applied multiplicatively to the complex FFT spectrum,
        # preserving phase information for the subsequent inverse FFT.
        #
        # Gaussian widths:
        #   ky direction: sigma = acc       (~10 rad/µm — tight, as ky is well-defined)
        #   kx direction: sigma = 10*acc    (~100 rad/µm — broader, to capture the full kx spread)
        print("Filtering in process...")

        mask = np.zeros(np.shape(DR), dtype=complex) + 1e-9   # Initialise with tiny non-zero floor

        for i in range(mask.shape[0]):
            for j in range(mask.shape[1]):
                # Multiply each k-space point by a 2-D Gaussian centred on (k_maxi, -k_peak).
                # Note: ky axis uses -k_peak because the scattered wave travels in -y direction.
                mask[i, j] = (fft[i, j]
                              * np.exp(-1 * ((kyy_i[i] + k_vec[peaks[k0]]) / acc)**2)       # Gaussian in ky
                              * np.exp(-1 * ((kxx_i[j] - k_maxi) / (10 * acc))**2))          # Gaussian in kx

        # Inverse fftshift before IFFT: undo the centring so k=0 is back at corner
        mask_i = np.fft.fftshift(mask, axes=(0, 1))

        # --- Plot the masked k-space spectrum ---
        mini = 1e-6;  maxi = 0.99 * DR.max()
        plt.figure(figsize=(4, 4))
        plt.title(f"{round(freq / 1e9, 3)} GHz")
        im = plt.imshow(np.abs(mask), origin="lower", aspect="auto",
                        extent=[np.amin(kxx_i) / d, np.amax(kxx_i) / d,
                                np.amin(kyy_i) / d, np.amax(kyy_i) / d],
                        norm=LogNorm(vmin=mini, vmax=maxi))
        plt.xlabel(r"$k_{x}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        plt.ylabel(r"$k_{y}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        # Filter boundary lines (commented; uncomment for visual verification)
        plt.axhline(kycut0 / d, c="magenta", lw=0.5, ls="--")
        plt.axhline(kycut1 / d, c="magenta", lw=0.5, ls="--")
        plt.axvline(kxcut0 / d, c="magenta", lw=0.5, ls="--")
        plt.axvline(kxcut1 / d, c="magenta", lw=0.5, ls="--")
        plt.show()


        # ── Step 6: inverse FFT → filtered real-space wave ───────────────────
        # The 2-D IFFT of the masked spectrum yields a complex-valued spatial field
        # whose amplitude |wave| represents the envelope of the isolated spin-wave mode.
        wave = np.fft.ifft2(mask_i)

        # wave = wave*(np.abs(wave)>np.abs(wave).max()*0.5)  # Optional hard threshold (commented)
        nn = 0.99   # Display saturation factor (cap colour scale at 99% of maximum)

        # --- Plot the filtered spin-wave amplitude in real space ---
        plt.figure(figsize=(6, 6))
        plt.title(r"$k_y$ = " + str(KY) + r" ($\frac{rad}{\mathrm{\mu m}}$)")
        im = plt.imshow(np.abs(wave), origin="lower", aspect="auto",
                        vmin=0, vmax=np.abs(wave).max() * nn,
                        extent=[x[x0:x1].min() * 1e6, x[x0:x1].max() * 1e6,
                                y[y0:y1].min() * 1e6, y[y0:y1].max() * 1e6],
                        cmap="magma")
        plt.xlabel(r"x ($\mathrm{\mu}$m)")
        plt.ylabel(r"y ($\mathrm{\mu}$m)")
        plt.colorbar(im, shrink=0.2)
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        # --- Save the filtered wave and spatial coordinates to disk ---
        # Filename convention: original data name + '_confl.npz' (confluence result)
        outname = (r"D:\mumax3\inelastic_scattering\{}\{}\filtered".format(k_dire, angle)
                   + data[:-13] + "_confl.npz")
        print(outname)
        np.savez(outname, x=x, y=y, wave=wave)   # Save: x coords, y coords, complex wave
        print("saved")
        print(" ");  print(" ")


# Convert accumulator lists to NumPy arrays for downstream analysis
Kx_vec_array = np.array(Kx_vec_array)   # kx components of all identified scattered waves (rad/µm)
Ky_vec_array = np.array(Ky_vec_array)   # ky components (rad/µm)
wavelength   = np.array(wavelength)     # Total wavelengths (m)

print("FINISHED")


In [ ]:
# Splitting process


In [ ]:
# =============================================================================
# Cell: Splitting (f_beam - f_edge) — k-space filtering batch loop
# =============================================================================
# Identical pipeline to the Confluence cell above, but targets the
# difference-frequency process:
#   f_split = f_SWB - f_edge
# In inelastic spin-wave scattering, this corresponds to stimulated splitting
# where the beam spin wave emits a magnon at f_edge and a daughter spin wave
# at the difference frequency propagates away from the scattering region.
# =============================================================================

# --- Batch control parameters (same structure as Confluence cell) ---
# Kx/Ky accumulators reset; k_dire can differ from the confluence run.
Kx_vec_array = []
Ky_vec_array = []
k_dire = 'k_pos'

f_Li   = np.arange(11.0, 15.51, 0.5, dtype=float)
amp    = "0.0300"

angle_Li = [35, 40, 45]   # Angles for this splitting batch

wavelength = []

# Active (narrowed) parameter set — overrides the broader lists above
angle_Li = np.array([32.5])
f_Li     = ["11.00", "11.50", "12.00", "12.50"]
k_dire   = 'k_pos'   # 'k_neg'


for angle in angle_Li:

    # Per-angle frequency overrides (commented; uncomment for targeted runs)
    #     if angle == 30:  f_Li = [ 13.75, 14.25, ... ]
    #     elif angle == 35: f_Li = [  12.75, 13.25, ... ]
    #     elif angle == 40: f_Li = [  11.75, 12.25, ... ]
    #     elif angle == 45: f_Li = [ 11.25, ... ]

    for f in f_Li:

        # --- Build file paths (same convention as Confluence) ---
        dir0     = r"D:\mumax3\inelastic_scattering\{}\{}\{}".format(k_dire, angle, amp)
        data     = r"\B_300mT_beam_f_45GHz_angle_{}_deg_edge_f_{}GHz_amp_{}_Mz_fzyxm.npz".format(angle, f, amp)
        dir_save = r'D:\mumax3\inelastic_scattering\{}\figs\{}\\'.format(k_dire, angle)
        path     = dir0 + data

        print("Opening: " + data)
        print("...")

        # --- Load and normalise the FFT data ---
        M_fzyxm = ovf.OvfFile(path)
        M_fzyxm.array /= (M_fzyxm.array).shape[0]   # Normalise by number of frequency bins
        M   = M_fzyxm.array
        fqs = M_fzyxm.time
        cy  = (M_fzyxm._headers)['ystepsize']
        cx  = (M_fzyxm._headers)['xstepsize']
        y   = np.arange(0, np.shape(M_fzyxm.array)[2], 1) * cy
        x   = np.arange(0, np.shape(M_fzyxm.array)[3], 1) * cx


        # ── Step 1: select the splitting (difference) frequency slice ────────
        # Splitting = difference-frequency process: f_split = f_SWB - f_edge
        freq     = f_SWB * 1e9 - float(f) * 1e9   # Target frequency (Hz) — note minus sign
        f_SWB_id = findNearest(fqs, freq)[0]
        M_SWB    = M_fzyxm.array[f_SWB_id, 0, :, :]   # 2-D slice; shape (ny, nx, n_comp) here
        colormap = np.abs(M_SWB)

        x0, x1 = 0, -1
        y0, y1 = 0, -1

        # --- Plot unfiltered real-space amplitude at splitting frequency ---
        plt.figure(figsize=(6, 6))
        plt.title("Splitting: " + data[:-23])
        im = plt.imshow(colormap[y0:y1, x0:x1], aspect='auto', origin="lower",
                        extent=[x[x0:x1].min() * 1e6, x[x0:x1].max() * 1e6,
                                y[y0:y1].min() * 1e6, y[y0:y1].max() * 1e6],
                        cmap="magma")
        plt.colorbar(im, shrink=0.3)
        plt.xlabel(r"x ($\mathrm{\mu m}$)")
        plt.ylabel(r"y ($\mathrm{\mu m}$)")
        # plt.savefig(...)   # Uncomment to save
        plt.show()


        # ── Step 2: 1-D edge profile → dominant ky identification ────────────
        # Same procedure as Confluence: extract real Mz along the right edge.
        # Note: M_SWB here has shape (ny, nx, n_comp), so index [:, -1, 0] is used
        # (component index 0 = Mz).
        if angle == 30:
            signal = np.real(M_SWB[:, -1, 0])
        else:
            signal = np.real(M_SWB[:, -1, 0])

        plt.title(path.split("/")[-1][:-23])
        plt.plot(y * 1e6, signal, c="r", lw=0.5)   # Red line to distinguish splitting from confluence
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        # 1-D FFT of the edge profile
        fft   = np.fft.fft(signal)
        k_vec = 2 * np.pi * np.fft.fftfreq(signal.shape[-1], y[1] - y[0])
        sh    = fft.shape[0] // 2

        # Peak detection with a lower threshold (33%) to catch weaker splitting signals
        peaks, _ = find_peaks(np.abs(fft)[:sh], height=np.abs(fft)[:sh].max() * 0.33)

        plt.title(path.split("/")[-1][:-23])
        plt.plot(k_vec[:sh] * 1e-6, np.abs(fft)[:sh], c="r", lw=0.5)
        plt.xlim(0, 300)
        nn = 1
        for i in range(np.shape(peaks)[0]):
            plt.axvline(k_vec[peaks[i]] * 1e-6, c="k", ls="--", lw=0.5,
                        label="Peak {}. = ".format(nn) + r"{}".format(
                            round(k_vec[peaks[i]] * 1e-6, 2)) + r" (rad/$\mathrm{\mu}$m)")
            nn += 1
        plt.legend()
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        print(k_vec[peaks])

        # Record ky (only the dominant peak for splitting)
        Ky_vec_array.append(k_vec[peaks[0]] * 1e-6)


        # ── Step 3: 2-D spatial FFT → k-space map ───────────────────────────
        # M_SWB[:,:,0] selects the Mz component (index 0) before the 2-D FFT.
        if angle == 30:
            fft_i = np.fft.fft2(M_SWB[y0:y1, x0:x1, 0])
        else:
            fft_i = np.fft.fft2(M_SWB[y0:y1, x0:x1, 0])

        fft = np.fft.fftshift(fft_i, axes=(0, 1))
        DR  = np.abs(fft)

        x_Li = x[x0:x1];  y_Li = y[y0:y1]
        dx   = x_Li[1] - x_Li[0];  dy = y_Li[1] - y_Li[0]

        kxx_i = np.fft.fftfreq(len(x_Li), dx) * 2 * pi
        kxx_i = np.concatenate((kxx_i[kxx_i.shape[0] // 2:], kxx_i[:kxx_i.shape[0] // 2]))
        kyy_i = np.fft.fftfreq(len(y_Li), dy) * 2 * pi
        kyy_i = np.concatenate((kyy_i[kyy_i.shape[0] // 2:], kyy_i[:kyy_i.shape[0] // 2]))


        # ── Step 4: locate (kx, ky) of the dominant scattered wave ───────────
        mini = 1e-2;  maxi = 0.99 * DR.max()
        d    = 1e6    # rad/m → rad/µm
        acc  = 10e6   # Gaussian filter half-width (rad/m)

        k0     = 0    # Use first detected peak
        kycut0 = -1 * k_vec[peaks[k0]] - acc
        kycut1 = -1 * k_vec[peaks[k0]] + acc
        KY     = round((np.abs(kycut0) + np.abs(kycut1)) / 2e6, 2)

        cut    = DR[findNearest(kyy_i, -1 * k_vec[peaks[k0]])[0], :]
        maxi   = findNearest(cut, cut.max())[0]
        k_maxi = kxx_i[maxi]

        Kx_vec_array.append(k_maxi * 1e-6)

        kxcut0 = k_maxi - acc
        kxcut1 = k_maxi + acc

        # Dispersion contour levels: splitting frequency and carrier
        freqLevels = [freq / 1e9, 45]

        # --- Plot k-space amplitude map with theoretical dispersion contours ---
        plt.figure(figsize=(6, 6))
        plt.title(r"splitting, f-$\nu$ = " + f"{round(freq / 1e9, 3)} GHz")
        im = plt.imshow(DR, origin="lower", aspect="auto",
                        norm=LogNorm(vmin=mini, vmax=0.99 * DR.max()),
                        extent=[np.amin(kxx_i) / d, np.amax(kxx_i) / d,
                                np.amin(kyy_i) / d, np.amax(kyy_i) / d])
        # Overlay Kalinikos-Slavin dispersion contours (red dashed) at the two relevant frequencies
        CS = plt.contour(kxx / d, kyy / d, fqs_2d * 1e-09, colors='red',
                         levels=freqLevels, linestyles='--', linewidths=0.5)
        plt.xlim(-250, 250);  plt.ylim(-250, 250)
        plt.xlabel(r"$k_{x}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        plt.ylabel(r"$k_{y}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        plt.axhline(-1 * k_vec[peaks[k0]] / d, c="magenta", lw=1.5, ls="--")   # Identified ky
        plt.axvline(k_maxi / d,                c="magenta", lw=1.5, ls="--")   # Identified kx
        # Filter boundary lines
        plt.axhline(kycut0 / d, c="magenta", lw=1.5, ls="--")
        plt.axhline(kycut1 / d, c="white",   lw=1.5, ls="--")
        plt.axvline(kxcut0 / d, c="magenta", lw=1.5, ls="--")
        plt.axvline(kxcut1 / d, c="white",   lw=1.5, ls="--")
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        # Accumulate wavelength of this scattered spin-wave mode
        wavelength.append((2 * np.pi) / np.sqrt(k_vec[peaks[k0]]**2 + k_maxi**2))


        # ── Step 5: apply 2-D Gaussian k-space mask ──────────────────────────
        # Identical Gaussian filtering approach as in the Confluence cell.
        # The narrow ky Gaussian (sigma=acc) and broad kx Gaussian (sigma=10*acc)
        # together isolate the dominant wavevector component while retaining the
        # kx spread of the beam.
        print("  ");  print("Filtering...");  print("  ")

        mask = np.zeros(np.shape(DR), dtype=complex) + 1e-9

        for i in range(mask.shape[0]):
            for j in range(mask.shape[1]):
                # Commented-out circular mask alternative (Euclidean distance in k-space):
                # r = np.sqrt((kxx_i[j]-k_maxi)**2 + (kyy_i[i]+k_vec[peaks[k0]])**2)
                mask[i, j] = (fft[i, j]
                              * np.exp(-1 * ((kyy_i[i] + k_vec[peaks[k0]]) / acc)**2)
                              * np.exp(-1 * ((kxx_i[j] - k_maxi) / (10 * acc))**2))

        mask_i = np.fft.fftshift(mask, axes=(0, 1))   # Undo centring before IFFT

        # --- Plot masked k-space spectrum ---
        mini = 1e-6;  maxi = 0.99 * DR.max()
        plt.figure(figsize=(4, 4))
        plt.title(f"{round(freq / 1e9, 3)} GHz")
        im = plt.imshow(np.abs(mask), origin="lower", aspect="auto",
                        extent=[np.amin(kxx_i) / d, np.amax(kxx_i) / d,
                                np.amin(kyy_i) / d, np.amax(kyy_i) / d],
                        norm=LogNorm(vmin=mini, vmax=maxi))
        plt.xlim(-250, 250);  plt.ylim(-250, 250)
        plt.xlabel(r"$k_{x}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        plt.ylabel(r"$k_{y}$ ($\frac{rad}{\mathrm{\mu m}}$)")
        # Filter boundary lines (commented; uncomment for verification)
        # plt.axhline(kycut0/d, ...); plt.axhline(kycut1/d, ...)
        # plt.axvline(kxcut0/d, ...); plt.axvline(kxcut1/d, ...)
        plt.show()


        # ── Step 6: inverse FFT → filtered real-space wave ───────────────────
        wave = np.fft.ifft2(mask_i)

        # wave = wave*(np.abs(wave)>np.abs(wave).max()*0.5)  # Optional hard threshold
        nn = 0.99

        plt.figure(figsize=(6, 6))
        plt.title(r"$k_y$ = " + str(KY) + r" ($\frac{rad}{\mathrm{\mu m}}$)")
        im = plt.imshow(np.abs(wave), origin="lower", aspect="auto",
                        vmin=0, vmax=np.abs(wave).max() * nn,
                        extent=[x[x0:x1].min() * 1e6, x[x0:x1].max() * 1e6,
                                y[y0:y1].min() * 1e6, y[y0:y1].max() * 1e6],
                        cmap="magma")
        plt.xlabel(r"x ($\mathrm{\mu}$m)")
        plt.ylabel(r"y ($\mathrm{\mu}$m)")
        plt.colorbar(im, shrink=0.2)
        # plt.savefig(...)   # Uncomment to save
        plt.show()

        # --- Save filtered wave to disk ---
        # Filename convention: original data name + '_split.npz' (splitting result)
        outname = (r"D:\mumax3\inelastic_scattering\{}\{}\filtered".format(k_dire, angle)
                   + data[:-13] + "_split.npz")
        print(outname)
        np.savez(outname, x=x, y=y, wave=wave)
        print("saved")
        print(" ");  print(" ")


print("DONE")

# Convert accumulator lists to NumPy arrays
Kx_vec_array = np.array(Kx_vec_array)
Ky_vec_array = np.array(Ky_vec_array)
wavelength   = np.array(wavelength)
